In [0]:
# %pip install sentence-transformers

In [0]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from datetime import datetime
from utils.model_helpers import *
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sentence_transformers import SentenceTransformer

In [0]:
# get the experiment id since mlflow.set_experiment() doesn't work with the explicit path (for some unknown reason)
# for exp in client.search_experiments():
#     print(f"id={exp.experiment_id}  name='{exp.name}'")

In [0]:
# ---- Config ----
CATALOG = "workspace"
LABELED_TABLE = f"{CATALOG}.careerpulse_gold.category_labeled_postings"
INFERENCE_TABLE = f"{CATALOG}.careerpulse_inference.category_predictions"
MLFLOW_EXPERIMENT_ID = "1902960386852945"
REGISTERED_MODEL = "careerpulse_category_knn"

# Cross-validation
CV_FOLDS = 5
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Candidate k values to sweep
K_CANDIDATES = [3, 5, 7, 9, 11]

mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)

In [0]:
# ---- Load labeled data from Gold ----
labeled_df = (
spark.table(LABELED_TABLE)
.select("posting_id", "title", "description_clean", "category")
.toPandas()
)
print(f"Total labeled rows : {len(labeled_df)}")
print(f"Unique categories : {labeled_df['category'].nunique()}")
print("\nClass distribution:")
print(labeled_df['category'].value_counts())

In [0]:
# ---- Feature: combined title + description text ----
# Concatenating title and description mirrors the approach validated in 03b.
# Title carries strong signal (short, dense) so prepending it boosts TF-IDF weights.
labeled_df["text"] = labeled_df["title"] + " " + labeled_df["description_clean"]
X = labeled_df["text"].values
y = labeled_df["category"].values
# Stratified train / test split — preserves class proportions in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)
print(f"Train size : {len(X_train)}")
print(f"Test size : {len(X_test)}")

In [0]:
# ---- CV helper ----
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [0]:
# ============================================================
# Approach 1: TF-IDF + KNN (cosine distance)
# Sweep over K_CANDIDATES — one MLflow run per k value.
# ============================================================

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")

tfidf_knn_results = []

for k in K_CANDIDATES:
    # create a new Pipeline set up for this run
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            sublinear_tf=True,
            min_df=2,
            ngram_range=(1, 2),
            max_features=50_000
            )),
        ("knn", KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
            ))
        ])
    # -- params
    params = {
        "embedding_method" : "tfidf",
        "n_neighbors" : k,
        "distance_metric" : "cosine",
        "tfidf_max_features": 50_000,
        "tfidf_ngram_range" : "(1,2)",
        "cv_folds" : CV_FOLDS,
        "test_size" : TEST_SIZE,
        }
    run_knn_experiment(
        estimator    = pipeline,
        X_train      = X_train,
        X_test       = X_test,
        y_train      = y_train,
        y_test       = y_test,
        cv           = cv,
        params       = params,
        results_list = tfidf_knn_results,
        run_name     = f"tfidf_knn_k{k}_{RUN_TIMESTAMP}"
    )

In [0]:
# ============================================================
# Approach 2: TF-IDF + SVD (LSA) + KNN (cosine distance)
# Normalizer re-unit-normalises after SVD so cosine distance
# is meaningful in the reduced space.
# ============================================================

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
SVD_COMPONENTS = 200 # Validated in 03b — captures most variance cheaply

tfidf_svd_knn_results = []

for k in K_CANDIDATES:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            sublinear_tf=True,
            min_df=2,
            ngram_range=(1, 2),
            max_features=50_000
            )),
        ("svd", TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)),
        ("norm", Normalizer(copy=False)),
        ("knn", KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
        ))
        ])
    params = {
            "embedding_method" : "tfidf_svd",
            "n_neighbors" : k,
            "distance_metric" : "cosine",
            "svd_components" : SVD_COMPONENTS,
            "tfidf_max_features": 50_000,
            "tfidf_ngram_range" : "(1,2)",
            "cv_folds" : CV_FOLDS,
            "test_size" : TEST_SIZE,
            }
    run_knn_experiment(
        estimator    = pipeline,
        X_train      = X_train,
        X_test       = X_test,
        y_train      = y_train,
        y_test       = y_test,
        cv           = cv,
        params       = params,
        results_list = tfidf_svd_knn_results,
        run_name     = f"tfidf_svd_knn_k{k}_{RUN_TIMESTAMP}"
    )

In [0]:
# ============================================================
# Approach 3: Sentence Transformer + KNN (cosine distance)
# Embeddings are dense and fixed-size — no TF-IDF needed.
# Encoding the full corpus once outside the CV loop avoids
# re-encoding on every fold, which is expensive.
# ============================================================

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
ST_MODEL_NAME = "all-MiniLM-L6-v2" # Fast, strong baseline validated in 03b

print(f"Encoding corpus with {ST_MODEL_NAME} ...")

st_model = SentenceTransformer(ST_MODEL_NAME)
X_encoded = st_model.encode(X, show_progress_bar=True, batch_size=64)

# NOTE: train_test_split shuffles indices, so we re-derive the split
# directly from the indices to keep alignment with y_train / y_test.
idx = np.arange(len(X))
idx_train, idx_test = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
X_enc_train = X_encoded[idx_train]
X_enc_test = X_encoded[idx_test]

st_knn_results = []

for k in K_CANDIDATES:
    knn = KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
            )
    params = {
        "embedding_method" : "sentence_transformer",
        "st_model" : ST_MODEL_NAME,
        "n_neighbors" : k,
        "distance_metric" : "cosine",
        "cv_folds" : CV_FOLDS,
        "test_size" : TEST_SIZE,
        }
    run_knn_experiment(
        estimator    = knn,
        X_train      = X_enc_train,
        X_test       = X_enc_test,
        y_train      = y_train,
        y_test       = y_test,
        cv           = cv,
        params       = params,
        results_list = st_knn_results,
        run_name     = f"sentence_transformer_knn_k{k}_{RUN_TIMESTAMP}"
    )

In [0]:
# ---- Comparison summary ----
all_results = pd.DataFrame(
    tfidf_knn_results + tfidf_svd_knn_results + st_knn_results
    )

print("\n=== CV F1-macro by method and k ===")

pivot = all_results.pivot_table(
    index="k",
    columns="embedding_method",
    values="cv_f1_macro_mean"
    )
print(pivot.to_string())

best_row = all_results.loc[all_results["cv_f1_macro_mean"].idxmax()]
print(f"\nBest config: {best_row['embedding_method']} k={int(best_row['k'])} "
f"cv_f1={best_row['cv_f1_macro_mean']:.4f}")

In [0]:
# ============================================================
# Approach 4: Sentence Transformer + KNN (cosine distance) + distance_weighting
# Weight nearest neighbors by distance so closer neighbors have more influence
# ============================================================

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")

st_distwts_knn_results = []

print("======== Sentence Transformer + KNN (cosine distance) + distance_weighting =====")
for k in K_CANDIDATES:
    knn = KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            weights="distance",
            n_jobs=-1
            )
    params = {
        "embedding_method" : "sentence_transformer",
        "st_model" : ST_MODEL_NAME,
        "n_neighbors" : k,
        "distance_metric" : "cosine",
        "weights": "distance",
        "title_prepending": 1,
        "cv_folds" : CV_FOLDS,
        "test_size" : TEST_SIZE,
        }
    run_knn_experiment(
        estimator    = knn,
        X_train      = X_enc_train,
        X_test       = X_enc_test,
        y_train      = y_train,
        y_test       = y_test,
        cv           = cv,
        params       = params,
        results_list = st_knn_results,
        run_name     = f"st_dist_wts_knn_k{k}_{RUN_TIMESTAMP}"
    )

In [0]:
# ============================================================
# Approach 5: Sentence Transformer + KNN (cosine distance) + Title Boosting
# Titles have a strong signal for category so prepending more than once
# should boost the signal and improve model results
# ============================================================

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
ST_MODEL_NAME = "all-MiniLM-L6-v2" # Fast, strong baseline validated in 03b

labeled_df["boosted_text"] = labeled_df["title"] + " " + labeled_df["title"] + " " + labeled_df["description_clean"]
X_boosted = labeled_df["boosted_text"].values

print(f"Encoding corpus with {ST_MODEL_NAME} ...")

st_model = SentenceTransformer(ST_MODEL_NAME)
X_boost_encoded = st_model.encode(X_boosted, show_progress_bar=True, batch_size=64)

# NOTE: train_test_split shuffles indices, so we re-derive the split
# directly from the indices to keep alignment with y_train / y_test.
idx = np.arange(len(X))
idx_train, idx_test = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
X_boost_enc_train = X_boost_encoded[idx_train]
X_boost_enc_test = X_boost_encoded[idx_test]

st_boosted_knn_results = []


print("======== Sentence Transformer + KNN (cosine distance) + Title Boosting =====")
for k in K_CANDIDATES:
    knn = KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            # weights="distance",
            n_jobs=-1
            )
    params ={
        "embedding_method" : "sentence_transformer",
        "st_model" : ST_MODEL_NAME,
        "n_neighbors" : k,
        "distance_metric" : "cosine",
        "weights": None,
        "title_prepending": 2,
        "cv_folds" : CV_FOLDS,
        "test_size" : TEST_SIZE,
        }
    run_knn_experiment(
        estimator    = knn,
        X_train      = X_enc_train,
        X_test       = X_enc_test,
        y_train      = y_train,
        y_test       = y_test,
        cv           = cv,
        params       = params,
        results_list = st_knn_results,
        run_name     = f"st_boosted_knn_k{k}_{RUN_TIMESTAMP}"
    )

In [0]:
print("======== Sentence Transformer + KNN (cosine distance) + Title Boosting + Distance Weighting =====")

for k in K_CANDIDATES:
    knn = KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            weights="distance",
            n_jobs=-1
            )
    params = {
            "embedding_method" : "sentence_transformer",
            "st_model" : ST_MODEL_NAME,
            "n_neighbors" : k,
            "distance_metric" : "cosine",
            "weights": "distance",
            "title_prepending": 2,
            "cv_folds" : CV_FOLDS,
            "test_size" : TEST_SIZE,
            }
    run_knn_experiment(
        estimator    = knn,
        X_train      = X_boost_enc_train,
        X_test       = X_boost_enc_test,
        y_train      = y_train,
        y_test       = y_test,
        cv           = cv,
        params       = params,
        results_list = st_knn_results,
        run_name     = f"st_boosted_distwtd_knn_k{k}_{RUN_TIMESTAMP}"
    )

In [0]:
# ============================================================
# Register winning model in MLflow Model Registry
# Update the parameters below to reflect the winning config.
# ============================================================

# --- Winning config ---
WINNING_METHOD        = "sentence_transformer"  # "tfidf" | "tfidf_svd" | "sentence_transformer"
WINNING_K             = 7
WINNING_WEIGHTS       = "distance" # "uniform" | "distance"
WINNING_TITLE_REPEATS = 2
ST_MODEL_NAME         = "all-MiniLM-L6-v2"      # ignored if not sentence_transformer
SVD_COMPONENTS        = 200                      # ignored if not tfidf_svd

# --- Load and prepare labeled data ---
labeled_df = (
    spark.table(LABELED_TABLE)
    .select("posting_id", "title", "description_clean", "category")
    .filter(F.col("category").isNotNull())
    .toPandas()
)

boost_title = lambda x, n: (x["title"] + " ") * n + x["description_clean"]
labeled_df["text"] = boost_title(labeled_df, WINNING_TITLE_REPEATS)
X_text = labeled_df["text"].values
y     = labeled_df["category"].values

# --- Build estimator and encode features based on WINNING_METHOD ---
if WINNING_METHOD == "tfidf":
    estimator = Pipeline([
        ("tfidf", TfidfVectorizer(
            sublinear_tf=True, min_df=2,
            ngram_range=(1, 2), max_features=50_000
        )),
        ("knn", KNeighborsClassifier(
            n_neighbors=WINNING_K, metric="cosine",
            algorithm="brute", weights=WINNING_WEIGHTS, n_jobs=-1
        ))
    ])
    X_encoded = X_text  # pipeline handles encoding internally

elif WINNING_METHOD == "tfidf_svd":
    estimator = Pipeline([
        ("tfidf", TfidfVectorizer(
            sublinear_tf=True, min_df=2,
            ngram_range=(1, 2), max_features=50_000
        )),
        ("svd",  TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)),
        ("norm", Normalizer(copy=False)),
        ("knn",  KNeighborsClassifier(
            n_neighbors=WINNING_K, metric="cosine",
            algorithm="brute", weights=WINNING_WEIGHTS, n_jobs=-1
        ))
    ])
    X_encoded = X_text  # pipeline handles encoding internally

elif WINNING_METHOD == "sentence_transformer":
    st_model  = SentenceTransformer(ST_MODEL_NAME)
    X_encoded = st_model.encode(X_text, show_progress_bar=True, batch_size=64)
    estimator = KNeighborsClassifier(
        n_neighbors=WINNING_K, metric="cosine",
        algorithm="brute", weights=WINNING_WEIGHTS, n_jobs=-1
    )

else:
    raise ValueError(f"Unknown WINNING_METHOD: '{WINNING_METHOD}'")

# --- Train/test split for honest evaluation ---
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# --- Evaluate on held-out test set ---
estimator.fit(X_train, y_train)
preds = estimator.predict(X_test)
final_metrics = {
    "test_accuracy" : float(accuracy_score(y_test, preds)),
    "test_f1_macro" : float(f1_score(y_test, preds, average="macro", zero_division=0)),
}
report = classification_report(y_test, preds, zero_division=0)
print(report)

# --- Refit on full labeled dataset ---
estimator.fit(X_encoded, y)

# --- Log and register with MLflow ---
with mlflow.start_run(run_name=f"WINNING_{WINNING_METHOD}_k{WINNING_K}_{RUN_TIMESTAMP}") as run:
    mlflow.log_params({
        "embedding_method"  : WINNING_METHOD,
        "n_neighbors"       : WINNING_K,
        "distance_metric"   : "cosine",
        "weights"           : WINNING_WEIGHTS,
        "title_prepending"  : WINNING_TITLE_REPEATS,
        "st_model"          : ST_MODEL_NAME if WINNING_METHOD == "sentence_transformer" else None,
        "svd_components"    : SVD_COMPONENTS if WINNING_METHOD == "tfidf_svd" else None,
        "train_size"        : len(X_train),
        "test_size"         : len(X_test),
    })
    mlflow.log_metrics(final_metrics)
    mlflow.log_text(report, artifact_file="classification_report.txt")

    model_info = mlflow.sklearn.log_model(
        sk_model=estimator,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL
    )

    print(f"Model registered : {REGISTERED_MODEL}")
    print(f"Run ID           : {run.info.run_id}")
    print(f"Model URI        : {model_info.model_uri}")